# OpenVLA — Step 2: SageMaker Training Job

This notebook launches a SageMaker training job to fine-tune OpenVLA-7B with LoRA on the BridgeData V2 dataset.

**Prerequisites:**
- Run `01_data_preparation.ipynb` first (dataset must be in S3)
- HuggingFace token with access to `openvla/openvla-7b`

**Instance:** ml.g6e.48xlarge (8× L40S 48GB) or ml.p4d.24xlarge (8× A100 40GB)

**Training time:** ~2-6 hours for 10 epochs

## 1. Setup

In [1]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('Enter your Hugging Face token: ')
login(token=hf_token)

Enter your Hugging Face token:  ········


In [2]:
import os, boto3, sagemaker
from sagemaker.core.helper import session_helper

sagemaker_session = session_helper.Session()
region = sagemaker_session.boto_region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

print(f'Region: {region}')
print(f'Account: {account_id}')
print(f'Bucket: {bucket_name}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Account: 783764584149
Bucket: sagemaker-us-east-1-783764584149


## 2. Training Configuration

In [3]:
from sagemaker.core.training.configs import (
    SourceCode, Compute, InputData, OutputDataConfig,
    StoppingCondition, CheckpointConfig,
)
from sagemaker.train import ModelTrainer

instance_type = 'ml.g6e.48xlarge'
instance_count = 1
image_uri = '763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.7-gpu-py312'

if default_prefix:
    S3_PREFIX = f'{default_prefix}/openvla-finetuning/datasets/bridge_hf_synthetic'
else:
    S3_PREFIX = 'openvla-finetuning/datasets/bridge_hf_synthetic'

dataset_s3_uri = f's3://{bucket_name}/{S3_PREFIX}'
job_name = 'openvla-lora-finetune-bridge'

if default_prefix:
    output_path = f's3://{bucket_name}/{default_prefix}/openvla-finetuning/{job_name}'
else:
    output_path = f's3://{bucket_name}/openvla-finetuning/{job_name}'

print(f'Instance: {instance_type} x {instance_count}')
print(f'Dataset: {dataset_s3_uri}')
print(f'Output: {output_path}')

Instance: ml.g6e.48xlarge x 1
Dataset: s3://sagemaker-us-east-1-783764584149/openvla-finetuning/datasets/bridge_hf_synthetic
Output: s3://sagemaker-us-east-1-783764584149/openvla-finetuning/openvla-lora-finetune-bridge


## 3. Create ModelTrainer

In [9]:
env = {
    'HF_TOKEN': hf_token,
    'ACCELERATE_CONFIG': './accelerate_configs/ddp.yaml',
    'training_recipe': './recipes/openvla_config.yaml',
    'FI_PROVIDER': 'efa',
    'NCCL_PROTO': 'simple',
    'NCCL_SOCKET_IFNAME': 'eth0',
    'NCCL_IB_DISABLE': '1',
    'NCCL_DEBUG': 'WARN',
}

source_code = SourceCode(
    source_dir='./scripts',
    requirements='requirements.txt',
    entry_script='run_finetuning.sh',
)

compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=3600,
)

model_trainer = ModelTrainer(
    training_image=image_uri,
    environment=env,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=36000),
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + '/checkpoints',
        local_path='/opt/ml/checkpoints',
    ),
)
print('ModelTrainer configured.')

[04/16/26 16:34:43] INFO     SageMaker session not provided. Using default Session.                  ]8;id=575554;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=659364;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Role not provided. Using default role:                                  ]8;id=739213;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=539828;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#75\75]8;;\
                             arn:aws:iam::783764584149:role/service-role/AmazonSageMaker-ExecutionRo               
                             le-20241230T144802                                                                    

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=535026;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=318615;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=889088;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=583465;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#553\553]8;;\
                             763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.7                     
                             -gpu-py312                                                                            

ModelTrainer configured.


## 4. Configure Input Data

In [10]:
train_input = InputData(channel_name='train', data_source=dataset_s3_uri)
data = [train_input]
print(f"Input channel 'train': {dataset_s3_uri}")

Input channel 'train': s3://sagemaker-us-east-1-783764584149/openvla-finetuning/datasets/bridge_hf_synthetic


## 5. Launch Training Job

In [11]:
print(f'Launching training job: {job_name}')
print(f'Instance: {instance_type} x {instance_count}')
model_trainer.train(input_data_config=data, wait=False)
print(f'Output will be at: {output_path}')

Launching training job: openvla-lora-finetune-bridge
Instance: ml.g6e.48xlarge x 1


[04/16/26 16:34:49] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=410023;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=375965;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#101\101]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[04/16/26 16:34:50] INFO     Creating training_job resource.                                     ]8;id=367027;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=675202;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

[04/16/26 16:34:51] WARNING  Not displaing the training container logs as 'wait' is set to     ]8;id=607408;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=148561;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#795\795]8;;\
                             False.                                                                                

Output will be at: s3://sagemaker-us-east-1-783764584149/openvla-finetuning/openvla-lora-finetune-bridge


## 6. (Optional) Check Status & Download Model

In [15]:
sm_client = boto3.client('sagemaker')
response = sm_client.list_training_jobs(
    NameContains=job_name, SortBy='CreationTime', SortOrder='Descending', MaxResults=5)
for job in response['TrainingJobSummaries']:
    print(f"{job['TrainingJobName']}: {job['TrainingJobStatus']}")

openvla-lora-finetune-bridge-20260416163449: Completed
openvla-lora-finetune-bridge-20260415224151: Failed


In [17]:
import boto3
sm_client = boto3.client('sagemaker')
response = sm_client.list_training_jobs(
    NameContains='openvla-lora-finetune-bridge',
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=5
)
for job in response['TrainingJobSummaries']:
    print(f"{job['TrainingJobName']}: {job['TrainingJobStatus']}")


openvla-lora-finetune-bridge-20260416163449: Completed
openvla-lora-finetune-bridge-20260415224151: Failed


In [18]:
import tarfile, boto3, os

sm_client = boto3.client('sagemaker')
job_name = 'openvla-lora-finetune-bridge'

response = sm_client.list_training_jobs(
    NameContains=job_name, StatusEquals='Completed',
    SortBy='CreationTime', SortOrder='Descending', MaxResults=1)

if not response['TrainingJobSummaries']:
    print('No completed training jobs found. Trying without status filter...')
    response = sm_client.list_training_jobs(
        NameContains=job_name, SortBy='CreationTime', SortOrder='Descending', MaxResults=5)
    for job in response['TrainingJobSummaries']:
        print(f"  {job['TrainingJobName']}: {job['TrainingJobStatus']}")

if not response['TrainingJobSummaries'] or response['TrainingJobSummaries'][0]['TrainingJobStatus'] != 'Completed':
    print('No completed job to download.')
else:
    completed_job = response['TrainingJobSummaries'][0]['TrainingJobName']
    job_desc = sm_client.describe_training_job(TrainingJobName=completed_job)
    model_s3_uri = job_desc['ModelArtifacts']['S3ModelArtifacts']
    print(f'Job: {completed_job}')
    print(f'Model artifacts: {model_s3_uri}')

    local_tar = f'./model_artifacts/{completed_job}/model.tar.gz'
    local_model_dir = f'./model_artifacts/{completed_job}/extracted/'
    os.makedirs(os.path.dirname(local_tar), exist_ok=True)
    os.makedirs(local_model_dir, exist_ok=True)

    s3_parts = model_s3_uri.replace('s3://', '').split('/', 1)
    print('Downloading...')
    boto3.client('s3').download_file(s3_parts[0], s3_parts[1], local_tar)
    print('Extracting...')
    with tarfile.open(local_tar, 'r:gz') as tar:
        tar.extractall(path=local_model_dir)
    print(f'Model extracted to: {local_model_dir}')
    print('You can now run 03_evaluation.ipynb')

No completed training jobs found. Trying without status filter...
  openvla-lora-finetune-bridge-20260416163449: Completed
  openvla-lora-finetune-bridge-20260415224151: Failed
Job: openvla-lora-finetune-bridge-20260416163449
Model artifacts: s3://sagemaker-us-east-1-783764584149/openvla-finetuning/openvla-lora-finetune-bridge/openvla-lora-finetune-bridge-20260416163449/output/model.tar.gz
Downloading...
Extracting...


/tmp/ipykernel_1793/392156907.py:36: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=local_model_dir)


Model extracted to: ./model_artifacts/openvla-lora-finetune-bridge-20260416163449/extracted/
You can now run 03_evaluation.ipynb
